In [ ]:
import json
import shutil
import subprocess
import time
from concurrent.futures import ThreadPoolExecutor, as_completed
from datetime import datetime
from pathlib import Path
 
import polars as pl
from Bio import SeqIO

In [ ]:
CWD = Path().resolve()
BASE = CWD
 
print(f"Folder where groups (data) folder should be: {BASE}")

In [ ]:
groups_map = {
    "plants_plt": ["solanaceae"],
    #"plants_mit": ["solanaceae"],
}
 
RUN_NAME = "clustalo_2026-10-08"  # output goes to <subgroup>/alignments/<RUN_NAME>/
CLUSTALO = "clustalo"
RE_RUN = False  # True re-aligns every IGR and allows changing the settings of an existing run
WORKERS = 32  # one IGR per worker, Clustal Omega with 1 thread each
 
if shutil.which(CLUSTALO) is None:
    raise FileNotFoundError(f"'{CLUSTALO}' not found in PATH")
clustalo_version = subprocess.run([CLUSTALO, "--version"], capture_output=True, text=True).stdout.strip()
print(f"Clustal Omega: {clustalo_version}")

In [ ]:
def log(msg: str) -> None:
    """Print a timestamped message."""
    print(f"[{datetime.now():%Y-%m-%d %H:%M:%S}] {msg}")
 
 
def build_cmd(in_fasta, out_fasta, log_file) -> list[str]:
    return [
        CLUSTALO,
        "-i", str(in_fasta),
        "-o", str(out_fasta),
        "--outfmt=fa",
        "-t", "DNA",
        "--full-iter",
        "--iter=3",
        "--output-order=input-order",
        "--threads=1",
        f"--log={log_file}",
        "--verbose",
        "--force",
    ]
 
 
def check_run_settings(run_json: Path, settings: dict) -> None:
    """Stop if RUN_NAME already exists with different settings (unless RE_RUN)."""
    if not run_json.exists():
        return
    old = json.loads(run_json.read_text())
    changed = {k: (old.get(k), v) for k, v in settings.items() if old.get(k) != v}
    if changed and not RE_RUN:
        details = "; ".join(f"{k}: {a!r} -> {b!r}" for k, (a, b) in changed.items())
        raise RuntimeError(
            f"{run_json.parent.name} already exists with different settings ({details}). "
            "Use a new RUN_NAME, or set RE_RUN = True."
        )
 
 
def run_clustalo(gp, igr_dir: Path, run_dir: Path, logs_dir: Path, force_run: bool) -> tuple:
    in_fasta = igr_dir / f"gp_{gp}.fasta"
    out_fasta = run_dir / f"gp_{gp}_aligned.fasta"
    log_file = logs_dir / f"gp_{gp}_clustalo.log"
 
    if out_fasta.exists() and not force_run:
        return gp, "skipped"
 
    tmp = out_fasta.with_suffix(".tmp")  # never leave a half-written alignment behind
    res = subprocess.run(build_cmd(in_fasta, tmp, log_file), stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    if res.returncode != 0 or not tmp.exists():
        (logs_dir / f"gp_{gp}_clustalo_error.log").write_text(res.stdout)
        tmp.unlink(missing_ok=True)
        return gp, f"error (exit {res.returncode})"
    tmp.replace(out_fasta)
    return gp, "ok"

In [ ]:
for g, subgs in groups_map.items():
    for subg in subgs:
        subg_dir = BASE / g / subg
        log(f"Processing {g}/{subg} | run: {RUN_NAME}")
 
        gp_file = subg_dir / "gene_pairs_filtered.tsv"
        if not gp_file.exists():
            log(f"  no {gp_file.name}, skipping")
            continue
 
        igr_dir = subg_dir / "igr_fastas"
        run_dir = subg_dir / "alignments" / RUN_NAME
        logs_dir = run_dir / "logs"
        logs_dir.mkdir(parents=True, exist_ok=True)
        run_json = run_dir / "run.json"
 
        template = build_cmd("<igr_fastas>/gp_{gp}.fasta", "<run_dir>/gp_{gp}_aligned.fasta", "<logs_dir>/gp_{gp}_clustalo.log")
        settings = {"command_template": " ".join(template)}
        check_run_settings(run_json, settings)
 
        # --- every listed IGR set needs an input FASTA with at least 2 sequences
        gp_df = pl.read_csv(gp_file, separator="\t")
        problems = []
        for gp in gp_df["gene_pair"]:
            fasta = igr_dir / f"gp_{gp}.fasta"
            if not fasta.exists():
                problems.append(f"{gp}: missing {fasta.name}")
            elif sum(1 for _ in SeqIO.parse(fasta, "fasta")) < 2:
                problems.append(f"{gp}: fewer than 2 sequences")
        if problems:
            log(f"  {len(problems)} of {gp_df.height} IGR sets have problems:")
            for p in problems:
                print(f"    {p}")
            raise RuntimeError(f"{g}/{subg}: input FASTAs are missing or too small; nothing was aligned.")
 
        # largest jobs first, so the long ones don't end up running alone at the end
        pairs = gp_df.sort(pl.col("igr_count") * pl.col("mean_len"), descending=True)["gene_pair"].to_list()
 
        run_info = {
            "run_name": RUN_NAME,
            "group": g,
            "subgroup": subg,
            "status": "running",
            "date_started": f"{datetime.now():%Y-%m-%d %H:%M:%S}",
            "date_finished": None,
            **settings,
            "clustalo": CLUSTALO,
            "clustalo_path": shutil.which(CLUSTALO),
            "clustalo_version": clustalo_version,
            "input_fastas": str(igr_dir.relative_to(subg_dir)),
            "input_fastas_abs": str(igr_dir.resolve()),
            "gene_pairs_file": str(gp_file.relative_to(subg_dir)),
            "gene_pairs_file_abs": str(gp_file.resolve()),
            "n_gene_pairs": len(pairs),
        }
        run_json.write_text(json.dumps(run_info, indent=2))
 
        # --- align
        t0 = time.time()
        counts: dict[str, int] = {}
        errors = []
        with ThreadPoolExecutor(max_workers=WORKERS) as executor:
            tasks = [executor.submit(run_clustalo, gp, igr_dir, run_dir, logs_dir, RE_RUN) for gp in pairs]
            for i, future in enumerate(as_completed(tasks), 1):
                gp, status = future.result()
                key = "error" if status.startswith("error") else status
                counts[key] = counts.get(key, 0) + 1
                if key == "error":
                    errors.append(f"{gp}: {status}")
                    log(f"  [ERROR] {gp}: {status}")
                if i % 25 == 0 or i == len(tasks):
                    log(f"    {i}/{len(tasks)} done | {counts}")
 
        runtime = time.time() - t0
        run_info.update(
            status="complete" if not errors else "complete_with_errors",
            date_finished=f"{datetime.now():%Y-%m-%d %H:%M:%S}",
            runtime_sec=round(runtime, 1),
            counts=counts,
            errors=errors,
        )
        run_json.write_text(json.dumps(run_info, indent=2))
 
        stale = {p.name.removeprefix("gp_").removesuffix("_aligned.fasta") for p in run_dir.glob("gp_*_aligned.fasta")} - {
            str(gp) for gp in pairs
        }
        log(f"  {g}/{subg}: {counts} | runtime {int(runtime // 60)}m{int(runtime % 60):02d}s -> {run_dir.relative_to(subg_dir)}/")
        if stale:
            log(f"  note: {len(stale)} alignments in {run_dir.relative_to(subg_dir)}/ are for IGR sets no longer listed")
 
log(f"Done with all groups/subgroups for run {RUN_NAME}")

plants_plt
    runtime: 55m27s
    pairs: 195

plants_mit
    runtime: 7m8s
    pairs: 32